# Negociação e compromisso entre agentes — exemplo

Passo a passo dos códigos da Aula 03 (Módulo 4): uma **rodada de contratação** em que três participantes propõem fazer a tarefa T1 do pedido P001.
O fluxo é: **anúncio → filtro das propostas → seleção do menor custo → compromisso**.

In [ ]:
# Rode esta célula apenas uma vez para instalar tudo na sua venv atual
!pip install langchain==1.4.7 langchain-core==1.7.1 pydantic==2.13.5 langchain-ollama==1.1.0 langgraph==1.2.18 ollama==0.6.3

## Passo 1 — O anúncio da rodada

O anúncio define as condições da tarefa: o requisito (estoque), o **limite** de chegada das propostas e o **orçamento** máximo.

In [ ]:
anuncio = {
    "pedido_id": "P001",
    "tarefa": "T1",
    "requisito": "estoque",
    "limite": 2,
    "orcamento": 10
}

## Passo 2 — As três propostas

Cada participante tem uma capacidade, um custo e um instante de chegada da proposta (valores didáticos do slide).

In [ ]:
capacidades = {
    "Ana": ["estoque"],
    "Bruno": ["logistica"],
    "Carla": ["estoque", "logistica"],
}

ofertas = [
    {"nome": "Ana",   "custo": 6, "chegada": 1},
    {"nome": "Bruno", "custo": 5, "chegada": 1},
    {"nome": "Carla", "custo": 4, "chegada": 3},
]

## Passo 3 — Filtrar antes de ordenar

Uma proposta só é válida se passar pelas **três condições obrigatórias**: capacidade, chegada dentro do limite e custo dentro do orçamento.
Um custo baixo não compensa a violação de uma condição.

In [ ]:
def filtrar_propostas(entrada):
    validas = [p for p in ofertas
        if entrada["requisito"] in capacidades[p["nome"]]
        and p["chegada"] <= entrada["limite"]
        and p["custo"] <= entrada["orcamento"]]
    return {**entrada, "validas": validas}

## Passo 4 — Selecionar a proposta

Entre as válidas, escolhe-se o **menor custo** (o nome desempata custos iguais).
Se a lista estiver vazia, o resultado é `None` — sem forçar um vencedor.

In [ ]:
def selecionar_proposta(rodada):
    if not rodada["validas"]:
        return None
    melhor = min(rodada["validas"],
        key=lambda p: (p["custo"], p["nome"]))
    return {"pedido_id": rodada["pedido_id"], "tarefa": "T1",
        "responsavel": melhor["nome"],
        "estado": "alocada", "custo": melhor["custo"]}

## Passo 5 — Montar o fluxo de negociação

A composição LangChain encadeia as etapas identificadas: `filtrar_propostas → selecionar_proposta`.

In [ ]:
from langchain_core.runnables import RunnableLambda

fluxo_negociacao = RunnableLambda(filtrar_propostas) | RunnableLambda(selecionar_proposta)

## Passo 6 — Rodada de referência (limite 2, orçamento 10)

- Bruno cai na capacidade (só logística).
- Carla cai na chegada (3 > 2).
- Ana é a única viável → **Ana** assume T1.

O aceite é um compromisso, não a conclusão da tarefa.

In [ ]:
fluxo_negociacao.invoke(anuncio)

## Passo 7 — Mesmo experimento com limite 3

Só o limite muda. Agora Ana (6) e Carla (4) são viáveis → a regra escolhe **Carla**.

In [ ]:
fluxo_negociacao.invoke({**anuncio, "limite": 3})

## Passo 8 — Rodada sem proposta viável (limite 2, orçamento 3)

- Ana: custo 6 excede o orçamento 3.
- Bruno: logística não atende estoque.
- Carla: chegada 3 excede o limite 2.

Resultado: `None`. Sem vencedor é um resultado válido; reabrir seria uma **nova rodada** com condições explícitas.

In [ ]:
resultado = fluxo_negociacao.invoke({**anuncio, "orcamento": 3})
print(resultado)

---
# Parte 2 — A mesma rodada com agentes de IA (Ollama)

Agora os participantes e o contratante são **agentes** (`create_agent`) rodando no Ollama.
A divisão de papéis segue o slide *Protocolo e estratégia*:

- **Protocolo (Python):** o filtro e a seleção continuam sendo as funções acima. A IA não escolhe o vencedor.
- **Estratégia (IA):** cada participante decide se **propõe ou recusa** e justifica; o contratante conduz a rodada e comunica o resultado.

## Passo 9 — Configurar o modelo

Mesma configuração usada em `envelops` e `agent-delegation`.

In [ ]:
from langchain_ollama import ChatOllama

config_base = dict(
    model='qwen3.5:9b',
    base_url='http://localhost:11434',
    reasoning=False,
    keep_alive='30m',
    num_ctx=8192,
    num_predict=1024,
)

# Temperatura baixa: decisões mais estáveis entre rodadas
modelo = ChatOllama(temperature=0.1, **config_base)

print('Modelo configurado!')

## Passo 10 — Função auxiliar para chamar um agente

Envia uma mensagem ao agente, mede o tempo e devolve só o texto da última resposta.

In [ ]:
import re
import time

def chamar(agente, conteudo: str, nome: str) -> str:
    inicio = time.perf_counter()
    resposta = agente.invoke(
        {'messages': [{'role': 'user', 'content': conteudo}]},
        config={'recursion_limit': 12}
    )
    texto = resposta['messages'][-1].content
    print(f"\n[{nome}] ({time.perf_counter() - inicio:.1f}s):\n{texto}")
    return texto

## Passo 11 — Os agentes participantes

Cada participante (Ana, Bruno, Carla) é um agente que recebe o anúncio e responde **PROPOR** ou **RECUSAR** com uma justificativa curta.
Os números da proposta (custo e chegada) vêm do perfil em Python: a IA decide *se* propõe, não altera os valores.

In [ ]:
from langchain.agents import create_agent

def criar_participante(nome: str):
    oferta = next(o for o in ofertas if o["nome"] == nome)
    return create_agent(modelo, tools=[], system_prompt=(
        f'Você é {nome}, um agente participante de uma rodada de contratação.\n'
        f'Seu perfil: capacidades = {capacidades[nome]}, custo da sua proposta = {oferta["custo"]}, '
        f'instante em que sua proposta chega = {oferta["chegada"]}.\n'
        'Ao receber um anúncio, decida se vale propor para a tarefa, considerando se você tem a capacidade pedida.\n'
        'Você NÃO pode mudar seu custo nem sua chegada.\n'
        'FORMATO DA RESPOSTA:\n'
        'Linha 1: exatamente PROPOR ou RECUSAR.\n'
        'Linha 2: uma justificativa curta e verificável, citando os dados do seu perfil.'
    ))

participantes = {nome: criar_participante(nome) for nome in capacidades}
print('Participantes:', list(participantes))

## Passo 12 — Filtro que usa as propostas recebidas

Na Parte 1 o filtro lia a lista fixa `ofertas`. Aqui ele lê só as propostas que os agentes **realmente enviaram** na rodada (`entrada["propostas"]`).
Também registramos **um motivo por participante** quando ele não passa (slide *Encerramento sem acordo*).

In [ ]:
def filtrar_propostas_rodada(entrada):
    validas = [p for p in entrada["propostas"]
        if entrada["requisito"] in capacidades[p["nome"]]
        and p["chegada"] <= entrada["limite"]
        and p["custo"] <= entrada["orcamento"]]
    return {**entrada, "validas": validas}

def motivos(entrada):
    """Explica, para cada participante, por que ele ficou de fora (ou que é viável)."""
    enviados = {p["nome"] for p in entrada["propostas"]}
    saida = {}
    for o in ofertas:
        nome = o["nome"]
        if nome not in enviados:
            saida[nome] = "recusou participar"
        elif entrada["requisito"] not in capacidades[nome]:
            saida[nome] = f"não atende ao requisito {entrada['requisito']}"
        elif o["chegada"] > entrada["limite"]:
            saida[nome] = f"chegada {o['chegada']} excede limite {entrada['limite']}"
        elif o["custo"] > entrada["orcamento"]:
            saida[nome] = f"custo {o['custo']} excede orçamento {entrada['orcamento']}"
        else:
            saida[nome] = "viável"
    return saida

fluxo_negociacao_ia = RunnableLambda(filtrar_propostas_rodada) | RunnableLambda(selecionar_proposta)

## Passo 13 — Ferramentas do contratante

O contratante não decide nada "de cabeça": ele usa duas ferramentas.

- `anunciar_rodada`: envia o anúncio a cada participante e guarda quem propôs.
- `fechar_rodada`: roda o protocolo em Python (filtro → seleção) e devolve o compromisso **ou `None`**, com os motivos.

O estado da rodada fica em `estado`, assim o modelo não precisa copiar dados nos argumentos.

In [ ]:
from langchain_core.tools import tool

estado = {"anuncio": {}, "propostas": [], "respostas": {}, "compromisso": None, "fechada": False}

def decisao(texto: str) -> str:
    """Primeira palavra da resposta, sem markdown (ex.: '**PROPOR**' -> 'PROPOR')."""
    limpo = re.sub(r'^[^A-Za-z]+', '', texto)
    primeira = limpo.split(maxsplit=1)[0] if limpo else ''
    return re.sub(r'[^A-Z]', '', primeira.upper())

@tool
def anunciar_rodada() -> str:
    """Envia o anúncio da rodada a todos os participantes e coleta quem propõe ou recusa."""
    a = estado["anuncio"]
    texto_anuncio = (
        f"Anúncio do pedido {a['pedido_id']}, tarefa {a['tarefa']}: requisito = {a['requisito']}, "
        f"limite de chegada = {a['limite']}, orçamento = {a['orcamento']}."
    )
    estado["propostas"], estado["respostas"] = [], {}
    for nome, agente in participantes.items():
        resposta = chamar(agente, texto_anuncio, nome)
        estado["respostas"][nome] = resposta
        if decisao(resposta) == "PROPOR":
            estado["propostas"].append(next(o for o in ofertas if o["nome"] == nome))
    return (
        "Respostas recebidas:\n" + "\n".join(f"- {n}: {r}" for n, r in estado["respostas"].items())
        + "\n\nA rodada ainda NÃO foi fechada. Chame fechar_rodada agora."
    )

@tool
def fechar_rodada() -> str:
    """Aplica o protocolo (filtro e menor custo) às propostas recebidas e devolve o compromisso ou None."""
    entrada = {**estado["anuncio"], "propostas": estado["propostas"]}
    estado["compromisso"] = fluxo_negociacao_ia.invoke(entrada)
    estado["fechada"] = True
    return (
        f"Compromisso: {estado['compromisso']}\n"
        f"Motivos por participante: {motivos(entrada)}"
    )

## Passo 14 — O agente contratante

Ele chama as ferramentas na ordem do protocolo e escreve o resultado.
Regra importante: se o compromisso for `None`, ele **não pode** escolher alguém por conveniência (slide *Disponibilidade não equivale a compromisso*).

In [ ]:
contratante = create_agent(modelo, tools=[anunciar_rodada, fechar_rodada], system_prompt=(
    'Você é o Contratante de uma rodada de contratação entre agentes.\n'
    'Você não escolhe o vencedor por conta própria: use as ferramentas, uma por vez, nesta ordem:\n'
    '1. anunciar_rodada;\n'
    '2. fechar_rodada.\n'
    'Depois responda em português, de forma curta:\n'
    '- quem assumiu o compromisso e por qual custo, OU "Sem acordo" se o compromisso for None;\n'
    '- um motivo por participante, usando os motivos devolvidos pela ferramenta.\n'
    'Se o compromisso for None, NUNCA indique um responsável, mesmo que alguém tenha capacidade. '
    'Lembre que o aceite é um compromisso, não a conclusão da tarefa.'
))

def rodar(anuncio_rodada: dict, titulo: str):
    print(f"\n{'=' * 10} {titulo} {'=' * 10}")
    estado.update(anuncio=anuncio_rodada, propostas=[], respostas={}, compromisso=None, fechada=False)
    inicio = time.perf_counter()
    chamar(contratante, f"Conduza a rodada do pedido {anuncio_rodada['pedido_id']}.", "Contratante")
    if not estado["fechada"]:
        # O agente pulou a etapa: o protocolo é aplicado mesmo assim, para o registro não ficar errado
        print("\n[Aviso] O contratante não chamou fechar_rodada; aplicando o protocolo em Python.")
        fechar_rodada.invoke({})
    print(f"\nCompromisso registrado: {estado['compromisso']}")
    print(f"Tempo total: {time.perf_counter() - inicio:.1f}s")

## Passo 15 — Rodada de referência (limite 2, orçamento 10)

Esperado: Bruno tende a recusar (não tem estoque), Carla cai na chegada → **Ana** assume T1.

In [ ]:
rodar(anuncio, "Referência: limite 2, orçamento 10")

## Passo 16 — Nova rodada com limite 3

Só o limite muda. Esperado: **Carla** (custo 4) vence Ana (custo 6).

In [ ]:
rodar({**anuncio, "limite": 3}, "Nova rodada: limite 3")

## Passo 17 — Rodada sem acordo (limite 2, orçamento 3)

Esperado: compromisso `None` e um motivo por participante. O contratante não deve inventar um vencedor.

In [ ]:
rodar({**anuncio, "orcamento": 3}, "Sem acordo: orçamento 3")

---
# Parte 3 — Ofertas alternadas: as IAs negociam entre si

Na Parte 2 cada participante respondia **uma vez** ao contratante. Aqui dois agentes **conversam**: cada um lê o histórico e responde à última oferta do outro (slide *Ofertas alternadas*).

- **Carla (fornecedora)** quer receber o maior preço por T1, mas tem um **mínimo** privado.
- **Coordenador (comprador)** quer pagar o menor preço, mas tem um **teto** privado.
- A cada turno a parte da vez **aceita**, **contrapropõe** ou **encerra**.

O Python é o protocolo: alterna os turnos, guarda o histórico, limita o número de turnos e bloqueia jogadas que violam o limite de autoridade do próprio agente.

## Passo 18 — Faixa em que o acordo é possível

Só existe acordo se `mínimo do fornecedor ≤ preço ≤ teto do comprador` (slide 22).
Os agentes **não** veem essa faixa: cada um conhece só o próprio limite.

In [ ]:
def faixa_de_acordo(minimo_fornecedor: float, teto_comprador: float):
    if minimo_fornecedor <= teto_comprador:
        return (minimo_fornecedor, teto_comprador)
    return None

print("Teto 10:", faixa_de_acordo(8, 10))
print("Teto 7: ", faixa_de_acordo(8, 7))

## Passo 19 — Os dois negociadores

Cada agente recebe seu limite **privado** e responde em um formato fixo: ação, preço e uma mensagem para a outra parte.

In [ ]:
NOMES_NEGOCIACAO = {"fornecedor": "Carla", "comprador": "Coordenador"}

def criar_negociador(papel: str, limite: float):
    if papel == "fornecedor":
        papel_txt = (
            f"Você é Carla, fornecedora que quer executar a tarefa T1 do pedido P001.\n"
            f"Seu preço MÍNIMO é {limite} (informação privada: nunca revele).\n"
            "Objetivo: fechar pelo MAIOR preço possível. Fechar acordo é melhor do que não fechar, "
            "mas nunca aceite nem proponha abaixo do seu mínimo."
        )
    else:
        papel_txt = (
            f"Você é o Coordenador, comprador que precisa contratar a tarefa T1 do pedido P001.\n"
            f"Seu TETO é {limite} (informação privada: nunca revele).\n"
            "Objetivo: fechar pelo MENOR preço possível. Fechar acordo é melhor do que não fechar, "
            "mas nunca aceite nem proponha acima do seu teto."
        )
    return create_agent(modelo, tools=[], system_prompt=(
        papel_txt + "\n\n"
        "A cada turno você recebe o histórico da negociação e o preço que está na mesa.\n"
        "Ações possíveis:\n"
        "- CONTRAPROPOR: você propõe um novo preço;\n"
        "- ACEITAR: você aceita o preço que está na mesa;\n"
        "- ENCERRAR: você desiste da negociação.\n"
        "Faça concessões aos poucos: não pule direto para o seu limite.\n\n"
        "FORMATO DA RESPOSTA (exatamente 3 linhas, sem markdown):\n"
        "ACAO: CONTRAPROPOR, ACEITAR ou ENCERRAR\n"
        "PRECO: apenas o número (se ACEITAR, repita o preço da mesa; se ENCERRAR, escreva 0)\n"
        "MENSAGEM: uma frase curta para a outra parte"
    ))

## Passo 20 — Ler e validar cada jogada

`validar_jogada` é o protocolo: confere se a ação existe e se o preço respeita o limite **do próprio agente**.
Um aceite ou uma contraproposta fora do limite é bloqueado, mesmo que o modelo tente.

In [ ]:
def ler_numero(texto: str):
    m = re.search(r"\d+(?:[.,]\d+)?", texto or "")
    return float(m.group().replace(",", ".")) if m else None

def ler_jogada(texto: str) -> dict:
    campos = {}
    for linha in texto.splitlines():
        if ":" in linha:
            chave, valor = linha.split(":", 1)
            campos[re.sub(r"[^A-Z]", "", chave.upper())] = valor.strip()
    return {
        "acao": re.sub(r"[^A-Z]", "", campos.get("ACAO", "").upper()),
        "preco": ler_numero(campos.get("PRECO", "")),
        "mensagem": campos.get("MENSAGEM", ""),
    }

def validar_jogada(papel: str, limite: float, jogada: dict, preco_mesa):
    """Devolve None se a jogada é válida, ou o motivo do bloqueio."""
    acao = jogada["acao"]
    if acao == "ENCERRAR":
        return None
    if acao == "ACEITAR":
        if preco_mesa is None:
            return "não há preço na mesa para aceitar"
        preco = preco_mesa
    elif acao == "CONTRAPROPOR":
        preco = jogada["preco"]
        if preco is None:
            return "preço ilegível"
    else:
        return f"ação desconhecida ({acao or 'vazia'})"
    if papel == "fornecedor" and preco < limite:
        return f"preço {preco} abaixo do seu mínimo {limite}"
    if papel == "comprador" and preco > limite:
        return f"preço {preco} acima do seu teto {limite}"
    return None

## Passo 21 — A conversa entre os agentes

A cada turno, a parte da vez recebe o **histórico completo** (o que a outra disse e propôs) e responde.

- Jogada inválida → o protocolo devolve o erro e dá **uma** nova chance; se errar de novo, a negociação termina sem acordo.
- **ACEITAR** → acordo no preço da mesa e o compromisso é registrado.
- **ENCERRAR** ou fim dos turnos → sem acordo, com o motivo registrado. O resultado é `None`, sem forçar contratação.

In [ ]:
def montar_mensagem(papel, limite, historico, preco_mesa, turnos_restantes):
    linhas = [f"Turno {h['turno']} - {h['quem']}: {h['acao']} {h['preco'] if h['preco'] is not None else ''} | \"{h['mensagem']}\""
              for h in historico]
    texto = "HISTÓRICO:\n" + ("\n".join(linhas) if linhas else "(nenhuma oferta ainda)")
    if preco_mesa is None:
        texto += "\n\nVocê abre a negociação: faça a primeira proposta."
    else:
        texto += f"\n\nPREÇO NA MESA (proposto pela outra parte): {preco_mesa}"
    texto += f"\nTurnos restantes na negociação: {turnos_restantes}"
    regra = f"nunca proponha nem aceite abaixo de {limite}" if papel == "fornecedor" else f"nunca proponha nem aceite acima de {limite}"
    texto += f"\nLembrete privado: {regra}. Se não houver como chegar a um acordo dentro disso, use ENCERRAR."
    return texto

def negociar(minimo_fornecedor: float = 8, teto_comprador: float = 10, max_turnos: int = 6, titulo: str = ""):
    print(f"\n{'=' * 10} {titulo} {'=' * 10}")
    print("Faixa de acordo (os agentes não veem):", faixa_de_acordo(minimo_fornecedor, teto_comprador))

    limites = {"fornecedor": minimo_fornecedor, "comprador": teto_comprador}
    agentes = {papel: criar_negociador(papel, limite) for papel, limite in limites.items()}
    historico, preco_mesa, vez = [], None, "fornecedor"
    resultado = {"acordo": False, "preco": None, "motivo": f"limite de {max_turnos} turnos atingido"}

    for turno in range(1, max_turnos + 1):
        nome = NOMES_NEGOCIACAO[vez]
        mensagem = montar_mensagem(vez, limites[vez], historico, preco_mesa, max_turnos - turno + 1)
        jogada = ler_jogada(chamar(agentes[vez], mensagem, f"{nome} - turno {turno}"))

        erro = validar_jogada(vez, limites[vez], jogada, preco_mesa)
        if erro:
            print(f"[Protocolo] Jogada bloqueada: {erro}. Nova chance para {nome}.")
            mensagem += f"\n\n[PROTOCOLO] Sua jogada anterior foi bloqueada: {erro}. Responda de novo."
            jogada = ler_jogada(chamar(agentes[vez], mensagem, f"{nome} - turno {turno} (2ª chance)"))
            erro = validar_jogada(vez, limites[vez], jogada, preco_mesa)
            if erro:
                resultado["motivo"] = f"{nome} fez jogada inválida duas vezes: {erro}"
                break

        preco_jogada = preco_mesa if jogada["acao"] == "ACEITAR" else jogada["preco"]
        historico.append({"turno": turno, "quem": nome, "acao": jogada["acao"],
                          "preco": preco_jogada if jogada["acao"] != "ENCERRAR" else None,
                          "mensagem": jogada["mensagem"]})

        if jogada["acao"] == "ACEITAR":
            resultado = {"acordo": True, "preco": preco_mesa, "motivo": f"{nome} aceitou no turno {turno}"}
            break
        if jogada["acao"] == "ENCERRAR":
            resultado["motivo"] = f"{nome} encerrou a negociação no turno {turno}"
            break

        preco_mesa = jogada["preco"]
        vez = "comprador" if vez == "fornecedor" else "fornecedor"

    compromisso = None
    if resultado["acordo"]:
        compromisso = {"pedido_id": "P001", "tarefa": "T1", "responsavel": "Carla",
                       "estado": "alocada", "custo": resultado["preco"]}

    print("\nHistórico:")
    for h in historico:
        print(f"- turno {h['turno']} | {h['quem']}: {h['acao']} {h['preco'] if h['preco'] is not None else ''}")
    print("Motivo do encerramento:", resultado["motivo"])
    print("Compromisso:", compromisso)
    return {**resultado, "historico": historico, "compromisso": compromisso}

## Passo 22 — Negociação com acordo possível (mínimo 8, teto 10)

Esperado: as ofertas se aproximam turno a turno e o acordo sai entre 8 e 10.
Existir uma faixa comum **não obriga** as partes a aceitar: o resultado depende das concessões.

In [ ]:
negociacao_ok = negociar(minimo_fornecedor=8, teto_comprador=10, titulo="Mínimo 8, teto 10")

## Passo 23 — Negociação sem faixa comum (mínimo 8, teto 7)

As condições deixam de se cruzar. Esperado: **sem acordo** e compromisso `None`; o protocolo bloqueia qualquer aceite fora dos limites.

In [ ]:
negociacao_sem_faixa = negociar(minimo_fornecedor=8, teto_comprador=7, titulo="Mínimo 8, teto 7")